# 35 · Tipo de treino sem o efeito da falha de GPS

Refaz o notebook 34 (ativos, engajamento e churn por tipo de treino principal) sem o efeito da falha
de GPS do Android (04/05 a 28/06/2026), para ler comportamento.

Não dá para saber quem saiu *por causa* da falha. Duas formas de tirar o efeito foram testadas:

1. **Tirar as contas atingidas** (treino com `gps_ok == False` ou chamado `sobre_falha_gps`), seções 1 a 3.
   **Resultado: enviesado, não usar para churn.** Só é "atingida" quem ainda treinava em maio e junho,
   então tirar essas contas remove sobreviventes dos meses anteriores: o churn de abr-mai do Android
   que sobra vai a 21% no ar livre (iOS: 12%), e o Android "sem falha" ainda sai a 18% em jun-jul.
2. **Usar só o iOS**, sistema em que a falha não aconteceu (seção 4). **É a leitura adotada.**

Tipo principal = o mais frequente nos 90 dias antes da data de referência; churn = perdas ÷ ativas no
início do mês, de abr a set/2026.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")
NIVEIS = ["1. alto", "2. médio", "3. baixo", "4. possível churn"]
TIPOS = ["corrida", "funcional", "caminhada", "forca", "yoga", "pedal"]

ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "tipo", "duracao_min", "origem"])
assin = pd.read_parquet(P / "assinaturas_limpa.parquet", columns=["user_id", "plano_atual", "inicio", "ativa_no_corte", "renovacao_automatica"])
usuarios = pd.read_parquet(P / "usuarios_limpa.parquet", columns=["user_id", "sexo", "idade_no_corte", "sistema"])
eng = pd.read_parquet(P / "engajamento_corte.parquet")
painel = pd.read_parquet(P / "assinaturas_mes.parquet", columns=["mes", "user_id", "plano", "ativa_inicio_mes", "perdida_no_mes"])
print(len(ativ), "treinos |", len(assin), "assinaturas |", len(eng), "ativos com nível |", len(painel), "linhas do painel")

gps = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "gps_ok"])
tk = pd.read_parquet(P / "tickets_suporte_limpa.parquet", columns=["user_id", "sobre_falha_gps"])
com_falha = set(gps.loc[~gps.gps_ok, "user_id"]); com_chamado = set(tk.loc[tk.sobre_falha_gps, "user_id"])
ATINGIDAS = com_falha | com_chamado
print("contas com treino com falha de GPS:", len(com_falha), "| com chamado sobre GPS:", len(com_chamado), "| nas duas:", len(com_falha & com_chamado), "| atingidas (união):", len(ATINGIDAS))
print("sistema das atingidas:", usuarios[usuarios.user_id.isin(ATINGIDAS)].sistema.value_counts().to_dict())
painel_todo = painel.copy(); assin_todo = assin.copy()
for nome in ["assin", "painel", "eng"]:
    d = globals()[nome]; antes = len(d)
    globals()[nome] = d[~d.user_id.isin(ATINGIDAS)]
    print(f"{nome}: {antes} -> {len(globals()[nome])} linhas sem as contas atingidas")
at0 = assin_todo[assin_todo.ativa_no_corte]
print("assinantes ativos no corte:", len(at0), "-> sem atingidas:", int(assin.ativa_no_corte.sum()), "| atingidas ativas:", int(at0.user_id.isin(ATINGIDAS).sum()))
pj = painel_todo[(painel_todo.mes >= "2026-04-01") & painel_todo.ativa_inicio_mes]
print("abr-set/2026, assinatura-mês:", len(pj), "-> sem atingidas:", int((~pj.user_id.isin(ATINGIDAS)).sum()), "| perdas:", int(pj.perdida_no_mes.sum()), "-> sem atingidas:", int(pj[~pj.user_id.isin(ATINGIDAS)].perdida_no_mes.sum()))

def perfil_de_treino(ref, dias=90):
    # tipo principal e variedade nos `dias` até `ref` (inclusive)
    a = ativ[(ativ.data <= ref) & (ativ.data > ref - pd.Timedelta(days=dias))]
    g = a.groupby(["user_id", "tipo"]).agg(n=("data", "size"), minutos=("duracao_min", "sum")).reset_index()
    g = g.sort_values(["user_id", "n", "minutos"], ascending=[True, False, False])
    top = g.drop_duplicates("user_id").set_index("user_id")
    tot = g.groupby("user_id").n.sum()
    r = pd.DataFrame({"principal": top.tipo, "treinos_90d": tot, "peso_principal": top.n / tot, "n_tipos": g.groupby("user_id").tipo.nunique()})
    pr = g.pivot(index="user_id", columns="tipo", values="n").fillna(0).reindex(columns=TIPOS, fill_value=0)
    return r.join(pr.add_prefix("n_"))

1252488 treinos | 32583 assinaturas | 14753 ativos com nível | 327771 linhas do painel
contas com treino com falha de GPS: 4319 | com chamado sobre GPS: 1079 | nas duas: 890 | atingidas (união): 4508
sistema das atingidas: {'android': 4508}
assin: 32583 -> 28640 linhas sem as contas atingidas
painel: 327771 -> 275656 linhas sem as contas atingidas
eng: 14753 -> 11807 linhas sem as contas atingidas
assinantes ativos no corte: 14753 -> sem atingidas: 11807 | atingidas ativas: 2946
abr-set/2026, assinatura-mês: 84868 -> sem atingidas: 64733 | perdas: 5542 -> sem atingidas: 4565


## 1. Ativos no corte por tipo de treino

In [2]:
at = assin[assin.ativa_no_corte].merge(usuarios, on="user_id", how="left").merge(eng[["user_id", "nivel", "treinos_30d"]], on="user_id", how="left", validate="one_to_one")
pf = perfil_de_treino(CORTE)
at = at.join(pf, on="user_id")
print("assinantes ativos:", len(at), "| com treino nos últimos 90 dias:", int(at.principal.notna().sum()), "| sem treino:", int(at.principal.isna().sum()))
at["principal"] = at.principal.fillna("sem treino")
for t in TIPOS: at["n_" + t] = at["n_" + t].fillna(0)

d = at.principal.value_counts().rename("ativos").to_frame()
d["% dos ativos"] = d.ativos / d.ativos.sum() * 100
d["% anual"] = at.groupby("principal").plano_atual.apply(lambda s: (s == "anual").mean() * 100)
d["treinos 90d (média)"] = at.groupby("principal").treinos_90d.mean()
d["peso do principal %"] = at.groupby("principal").peso_principal.mean() * 100
d["nº de tipos (média)"] = at.groupby("principal").n_tipos.mean()
print("== tipo PRINCIPAL dos assinantes ativos"); print(d); print()

pr = pd.DataFrame({"ativos que praticam": [(at["n_" + t] > 0).sum() for t in TIPOS]}, index=TIPOS)
pr["% dos ativos"] = pr["ativos que praticam"] / len(at) * 100
pr["% dos que praticam e têm como principal"] = [((at.principal == t).sum() / (at["n_" + t] > 0).sum()) * 100 for t in TIPOS]
pr["treinos 90d do tipo (média de quem pratica)"] = [at.loc[at["n_" + t] > 0, "n_" + t].mean() for t in TIPOS]
print("== tipos que os ativos PRATICAM (uma conta entra em vários)"); print(pr); print()
print("nº de tipos praticados em 90 dias"); x = at.n_tipos.fillna(0).astype(int).value_counts().sort_index().rename("ativos").to_frame(); x["%"] = x.ativos / x.ativos.sum() * 100; print(x)

assinantes ativos: 11807 | com treino nos últimos 90 dias: 11143 | sem treino: 664
== tipo PRINCIPAL dos assinantes ativos
            ativos  % dos ativos  % anual  treinos 90d (média)  peso do principal %  nº de tipos (média)
principal                                                                                               
corrida       4059          34.4     57.7                 25.5                 75.0                  3.7
funcional     2949          25.0     50.7                 28.5                 46.7                  3.7
forca         1516          12.8     53.4                 23.6                 43.4                  3.6
caminhada     1388          11.8     49.6                 22.0                 67.8                  3.7
pedal         1004           8.5     53.1                 27.4                 73.8                  3.8
sem treino     664           5.6     99.5                  NaN                  NaN                  NaN
yoga           227           1.9     

In [3]:
print("== tipo principal por plano (% da coluna)"); print(pd.crosstab(at.principal, at.plano_atual, normalize="columns") * 100); print()
print("== tipo principal por sexo (% da coluna)"); print(pd.crosstab(at.principal, at.sexo.fillna("não informado"), normalize="columns") * 100); print()
at["faixa"] = pd.cut(at.idade_no_corte.astype(float), [0, 34, 44, 54, 120], labels=["até 34", "35-44", "45-54", "55+"]).astype(object).fillna("não informado")
print("== tipo principal por faixa de idade (% da coluna)"); print(pd.crosstab(at.principal, at.faixa, normalize="columns") * 100); print()
print("== combinações: quem tem o tipo da linha como principal também pratica (% da linha)")
print((at[at.principal != "sem treino"].groupby("principal")[["n_" + t for t in TIPOS]].agg(lambda s: (s > 0).mean() * 100)).rename(columns=lambda c: c[2:]))

== tipo principal por plano (% da coluna)
plano_atual  anual  mensal
principal                 
caminhada     10.3    13.6
corrida       35.1    33.4
forca         12.1    13.7
funcional     22.4    28.3
pedal          8.0     9.2
sem treino     9.9     0.1
yoga           2.0     1.8

== tipo principal por sexo (% da coluna)
sexo          F    M  não informado
principal                          
caminhada  11.8 11.9           10.2
corrida    34.5 34.0           36.6
forca      12.5 13.3           11.9
funcional  24.7 25.4           24.5
pedal       9.0  7.9            8.2
sem treino  5.6  5.6            5.9
yoga        1.9  1.9            2.7

== tipo principal por faixa de idade (% da coluna)
faixa       35-44  45-54  55+  até 34  não informado
principal                                           
caminhada    11.9   11.7 11.9    11.0           12.4
corrida      34.8   34.1 33.4    36.1           34.3
forca        12.8   13.2 12.5    12.6           13.0
funcional    24.4   25.8 25.8   

## 2. Engajamento dos ativos por tipo principal

In [4]:
n = pd.crosstab(at.principal, at.nivel).reindex(columns=NIVEIS, fill_value=0)
e = (n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("ativos")).join(at.groupby("principal").treinos_30d.mean().rename("treinos 30d (média)"))
e["% renovação desligada"] = at.groupby("principal").renovacao_automatica.apply(lambda s: (s == "nao").mean() * 100)
e = e.sort_values("ativos", ascending=False); print(e); print()
for pl in ["mensal", "anual"]:
    k = at[at.plano_atual == pl]
    n = pd.crosstab(k.principal, k.nivel).reindex(columns=NIVEIS, fill_value=0)
    print(pl); print((n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("ativos")).sort_values("ativos", ascending=False)); print()
at["variedade"] = pd.cut(at.n_tipos.fillna(0), [-1, 0, 1, 2, 3, 6], labels=["sem treino", "1 tipo", "2 tipos", "3 tipos", "4+ tipos"])
n = pd.crosstab(at.variedade, at.nivel).reindex(columns=NIVEIS, fill_value=0)
print("por variedade de tipos"); print((n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("ativos")))

            1. alto  2. médio  3. baixo  4. possível churn  ativos  treinos 30d (média)  % renovação desligada
principal                                                                                                     
corrida        74.0      18.6       2.6                4.8    4059                 10.0                   16.2
funcional      81.2      14.4       1.8                2.5    2949                 10.9                    8.9
forca          77.0      17.0       2.4                3.5    1516                  9.1                   12.9
caminhada      73.9      18.2       3.5                4.3    1388                  8.5                   16.0
pedal          77.9      15.7       2.7                3.7    1004                 10.7                   11.8
sem treino      0.0       0.8       0.0               99.2     664                  0.0                   32.8
yoga           65.6      15.9       4.4               14.1     227                  5.3                   26.9



## 3. Churn por tipo principal (abr a set/2026)

In [5]:
partes = []
for mes in pd.date_range("2026-04-01", "2026-09-01", freq="MS"):
    k = painel[(painel.mes == mes) & painel.ativa_inicio_mes]
    pf = perfil_de_treino(mes - pd.Timedelta(days=1))
    k = k.join(pf[["principal", "n_tipos", "treinos_90d"]], on="user_id")
    print(mes.date(), "ativas no início do mês:", len(k), "| sem treino nos 90 dias:", int(k.principal.isna().sum()), "| perdas:", int(k.perdida_no_mes.sum()))
    partes.append(k)
pm = pd.concat(partes, ignore_index=True)
pm["principal"] = pm.principal.fillna("sem treino")
pm["variedade"] = pd.cut(pm.n_tipos.fillna(0), [-1, 0, 1, 2, 3, 6], labels=["sem treino", "1 tipo", "2 tipos", "3 tipos", "4+ tipos"])
print("assinatura-mês:", len(pm))

def churn(d, por):
    g = d.groupby(por, observed=True).agg(ativas=("user_id", "size"), perdas=("perdida_no_mes", "sum"))
    g["churn %"] = g.perdas / g.ativas * 100
    return g
print(); print("== todos os planos"); c0 = churn(pm, "principal").sort_values("ativas", ascending=False); c0["% das perdas"] = c0.perdas / c0.perdas.sum() * 100; c0["% da base"] = c0.ativas / c0.ativas.sum() * 100; print(c0)
cp = churn(pm, ["plano", "principal"])
print(); print("== churn % por plano"); print(cp["churn %"].unstack(0).join(cp.ativas.unstack(0), rsuffix=" (ativas)").loc[c0.index])
print(); print("== por variedade de tipos"); cv = churn(pm, ["plano", "variedade"]); print(cv["churn %"].unstack(0).join(cv.ativas.unstack(0), rsuffix=" (ativas)"))
print(); print("== mensal: churn % por mês e tipo principal"); print(churn(pm[pm.plano == "mensal"], ["mes", "principal"])["churn %"].unstack()[c0.index])

2026-04-01 ativas no início do mês: 11078 | sem treino nos 90 dias: 593 | perdas: 1142


2026-05-01 ativas no início do mês: 10677 | sem treino nos 90 dias: 652 | perdas: 948


2026-06-01 ativas no início do mês: 10444 | sem treino nos 90 dias: 706 | perdas: 594
2026-07-01 ativas no início do mês: 10502 | sem treino nos 90 dias: 753 | perdas: 639


2026-08-01 ativas no início do mês: 10780 | sem treino nos 90 dias: 775 | perdas: 649
2026-09-01 ativas no início do mês: 11252 | sem treino nos 90 dias: 735 | perdas: 593


assinatura-mês: 64733

== todos os planos
            ativas  perdas  churn %  % das perdas  % da base
principal                                                   
corrida      22065    1749      7.9          38.3       34.1
funcional    15518     863      5.6          18.9       24.0
forca         8528     532      6.2          11.7       13.2
caminhada     7486     533      7.1          11.7       11.6
pedal         5597     437      7.8           9.6        8.6
sem treino    4214     342      8.1           7.5        6.5
yoga          1325     109      8.2           2.4        2.0

== churn % por plano
plano       anual  mensal  anual (ativas)  mensal (ativas)
principal                                                 
corrida       3.1    14.9           13049             9016
funcional     2.1     9.4            8090             7428
forca         2.9    10.1            4532             3996
caminhada     4.1    10.4            3910             3576
pedal         3.6    12.9        

In [6]:
# Mesmo volume de treino: o tipo ainda importa? (mensal)
m = pm[(pm.plano == "mensal") & (pm.principal != "sem treino")].copy()
m["volume"] = pd.cut(m.treinos_90d, [0, 12, 30, 1000], labels=["1-12 treinos", "13-30", "31+"])
x = churn(m, ["volume", "principal"])
print("MENSAL: churn % por volume de treino em 90 dias e tipo principal"); print(x["churn %"].unstack(0)); print(x.ativas.unstack(0))
print()
# ar livre (corrida, caminhada, pedal) contra indoor, por sistema, em jun-jul (falha de GPS do Android)
u = pm.merge(usuarios[["user_id", "sistema"]], on="user_id", how="left")
u["ambiente"] = np.where(u.principal.isin(["corrida", "caminhada", "pedal"]), "ar livre", np.where(u.principal == "sem treino", "sem treino", "indoor"))
u["periodo"] = np.where(u.mes.isin(pd.to_datetime(["2026-06-01", "2026-07-01"])), "jun-jul", "abr-mai e ago-set")
print("MENSAL: churn % por ambiente do tipo principal, sistema e período")
print(churn(u[u.plano == "mensal"], ["periodo", "ambiente", "sistema"])["churn %"].unstack())

# Conferência: só iOS (sem falha de GPS), com todas as contas do sistema
ios = set(usuarios.loc[usuarios.sistema == "ios", "user_id"])
partes = []
for mes in pd.date_range("2026-04-01", "2026-09-01", freq="MS"):
    k = painel_todo[(painel_todo.mes == mes) & painel_todo.ativa_inicio_mes & painel_todo.user_id.isin(ios)]
    partes.append(k.join(perfil_de_treino(mes - pd.Timedelta(days=1))[["principal", "n_tipos"]], on="user_id"))
pi = pd.concat(partes, ignore_index=True); pi["principal"] = pi.principal.fillna("sem treino")
pi["variedade"] = pd.cut(pi.n_tipos.fillna(0), [-1, 0, 1, 2, 3, 6], labels=["sem treino", "1 tipo", "2 tipos", "3 tipos", "4+ tipos"])
ci = churn(pi, ["plano", "principal"])
print(); print("== CONFERÊNCIA só iOS:", len(pi), "assinatura-mês | churn % por plano e tipo principal")
print(ci["churn %"].unstack(0).join(ci.ativas.unstack(0), rsuffix=" (ativas)").loc[c0.index])
print(); print("== lado a lado, MENSAL: sem atingidas × só iOS")
print(pd.DataFrame({"sem atingidas": cp["churn %"].unstack(0)["mensal"], "só iOS": ci["churn %"].unstack(0)["mensal"]}).loc[c0.index])
print(); print("só iOS, por variedade"); cvi = churn(pi, ["plano", "variedade"]); print(cvi["churn %"].unstack(0))

final = d.join(e[NIVEIS].add_prefix("% ")).join(c0[["churn %"]].rename(columns={"churn %": "churn mensal abr-set/26 %"})).join(cp["churn %"].unstack(0).add_prefix("churn % "))
final.to_csv(S / "tipo_treino_sem_falha_gps.csv", encoding="utf-8-sig")
print(); print(final.T)

MENSAL: churn % por volume de treino em 90 dias e tipo principal
volume     1-12 treinos  13-30  31+
principal                          
caminhada          13.9   10.6  7.5
corrida            18.2   15.4 13.0
forca              13.0    9.6  9.0
funcional          13.5    8.9  8.7
pedal              15.2   12.8 12.0
yoga               11.9    8.9 11.1
volume     1-12 treinos  13-30   31+
principal                           
caminhada           765   1758  1053
corrida            1518   3999  3499
forca               769   2072  1155
funcional           897   3490  3041
pedal               408   1140   999
yoga                201    248    54

MENSAL: churn % por ambiente do tipo principal, sistema e período
sistema                       android  ios
periodo           ambiente                
abr-mai e ago-set ar livre       21.1 11.7
                  indoor         12.7  8.8
                  sem treino     16.7 20.0
jun-jul           ar livre       18.3  9.3
                  indoor  


== CONFERÊNCIA só iOS: 41233 assinatura-mês | churn % por plano e tipo principal
plano       anual  mensal  anual (ativas)  mensal (ativas)
principal                                                 
corrida       2.6    11.8            8952             6587
funcional     2.0     8.0            4619             4399
forca         2.2     9.2            2543             2338
caminhada     3.0     8.7            2629             2497
pedal         3.1    10.5            2049             1881
sem treino    8.1    19.0            1986               21
yoga          6.1     7.9             441              291

== lado a lado, MENSAL: sem atingidas × só iOS
            sem atingidas  só iOS
principal                        
corrida              14.9    11.8
funcional             9.4     8.0
forca                10.1     9.2
caminhada            10.4     8.7
pedal                12.9    10.5
sem treino           17.1    19.0
yoga                 10.3     7.9

só iOS, por variedade
plano     

## 4. Leitura adotada: só iOS (ativos, engajamento e churn)

In [7]:
ai = assin_todo[assin_todo.ativa_no_corte & assin_todo.user_id.isin(ios)].merge(pd.read_parquet(P / "engajamento_corte.parquet")[["user_id", "nivel", "treinos_30d"]], on="user_id", how="left", validate="one_to_one")
ai = ai.join(perfil_de_treino(CORTE), on="user_id"); ai["principal"] = ai.principal.fillna("sem treino")
print("assinantes ativos no corte:", int(assin_todo.ativa_no_corte.sum()), "-> só iOS:", len(ai))
n = pd.crosstab(ai.principal, ai.nivel).reindex(columns=NIVEIS, fill_value=0)
ti = n.sum(axis=1).rename("ativos").to_frame(); ti["% dos ativos"] = ti.ativos / ti.ativos.sum() * 100
ti = ti.join((n.div(n.sum(axis=1), axis=0) * 100).add_prefix("% "))
ti["% renovação desligada"] = ai.groupby("principal").renovacao_automatica.apply(lambda s: (s == "nao").mean() * 100)
ti["treinos 30d (média)"] = ai.groupby("principal").treinos_30d.mean()
ti = ti.join(ci["churn %"].unstack(0).add_prefix("churn % ")).join(ci.perdas.unstack(0).add_prefix("perdas "))
ti = ti.sort_values("ativos", ascending=False); print(ti)
ti.to_csv(S / "tipo_treino_so_ios.csv", encoding="utf-8-sig")
print()
ai["variedade"] = pd.cut(ai.n_tipos.fillna(0), [-1, 0, 1, 2, 3, 6], labels=["sem treino", "1 tipo", "2 tipos", "3 tipos", "4+ tipos"])
n = pd.crosstab(ai.variedade, ai.nivel).reindex(columns=NIVEIS, fill_value=0)
print("iOS, por variedade"); print((n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("ativos")).join(cvi["churn %"].unstack(0).add_prefix("churn % ")))
print()
mi = pi[(pi.plano == "mensal") & (pi.principal != "sem treino")].join(pd.concat([perfil_de_treino(m - pd.Timedelta(days=1))[["treinos_90d"]].assign(mes=m) for m in pd.date_range("2026-04-01", "2026-09-01", freq="MS")]).set_index("mes", append=True), on=["user_id", "mes"])
mi["volume"] = pd.cut(mi.treinos_90d, [0, 12, 30, 1000], labels=["1-12 treinos", "13-30", "31+"])
x = churn(mi, ["volume", "principal"]); print("iOS MENSAL: churn % por volume de treino em 90 dias e tipo principal"); print(x["churn %"].unstack(0)); print(x.ativas.unstack(0))
print()
print("iOS MENSAL: churn % por mês e tipo principal"); print(churn(pi[pi.plano == "mensal"], ["mes", "principal"])["churn %"].unstack()[c0.index])
print()
# o mesmo com todas as contas (notebook 34), para comparar
partes = []
for mes in pd.date_range("2026-04-01", "2026-09-01", freq="MS"):
    k = painel_todo[(painel_todo.mes == mes) & painel_todo.ativa_inicio_mes]
    partes.append(k.join(perfil_de_treino(mes - pd.Timedelta(days=1))[["principal"]], on="user_id"))
pt = pd.concat(partes, ignore_index=True); pt["principal"] = pt.principal.fillna("sem treino")
ct = churn(pt, ["plano", "principal"])["churn %"].unstack(0)
print("MENSAL: todas as contas (nb 34) × só iOS"); print(pd.DataFrame({"todas": ct["mensal"], "só iOS": ci["churn %"].unstack(0)["mensal"], "anual todas": ct["anual"], "anual só iOS": ci["churn %"].unstack(0)["anual"]}).loc[c0.index])

assinantes ativos no corte: 14753 -> só iOS: 7278
            ativos  % dos ativos  % 1. alto  % 2. médio  % 3. baixo  % 4. possível churn  % renovação desligada  treinos 30d (média)  churn % anual  churn % mensal  perdas anual  perdas mensal
principal                                                                                                                                                                                       
corrida       2706          37.2       75.3        17.5         2.7                  4.5                   15.2                 10.3            2.6            11.8           232            780
funcional     1676          23.0       82.5        13.7         1.7                  2.1                    8.0                 11.2            2.0             8.0            94            352
caminhada      907          12.5       75.7        17.8         3.1                  3.4                   13.8                  9.0            3.0             8.7            80 

iOS MENSAL: churn % por volume de treino em 90 dias e tipo principal
volume     1-12 treinos  13-30  31+
principal                          
caminhada          14.7    8.8  5.0
corrida            14.4   12.4 10.4
forca              11.8    9.0  7.7
funcional          12.8    7.7  7.1
pedal              14.2    9.5 10.3
yoga                8.7    7.9  5.4
volume     1-12 treinos  13-30   31+
principal                           
caminhada           448   1245   804
corrida             974   2847  2766
forca               450   1164   724
funcional           467   2041  1891
pedal               261    851   769
yoga                115    139    37

iOS MENSAL: churn % por mês e tipo principal
principal   corrida  funcional  forca  caminhada  pedal  sem treino  yoga
mes                                                                      
2026-04-01     15.7       12.1   12.3       10.3   16.2        20.0  14.6
2026-05-01     15.7       10.4   12.0       12.1   12.3        33.3   8.3
2026-

MENSAL: todas as contas (nb 34) × só iOS
            todas  só iOS  anual todas  anual só iOS
principal                                           
corrida      13.8    11.8          2.7           2.6
funcional     8.6     8.0          1.8           2.0
forca         9.3     9.2          2.6           2.2
caminhada     9.8     8.7          3.5           3.0
pedal        12.0    10.5          2.9           3.1
sem treino   17.9    19.0          8.0           8.1
yoga          9.5     7.9          6.4           6.1
